In [2]:
# Modülde neler yapılacağını bul
# Trust İndex
# Performans İndex 
# WOEleyip portföy oluşturma

In [3]:
# QuiverQuant API key'i dosyadan oku
with open("Quiver API KEY.txt", "r") as f:
    API_KEY = f.read().strip()

headers = {"Authorization": f"Token {API_KEY}"}


In [ ]:
import pandas as pd
import requests

def list_slickcharts_sp500() -> pd.DataFrame:
    # Ref: https://stackoverflow.com/a/75845569/
    url = 'https://www.slickcharts.com/sp500'
    user_agent = 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/111.0'  # Default user-agent fails.
    response = requests.get(url, headers={'User-Agent': user_agent})
    return pd.read_html(response.text, match='Symbol', index_col='Symbol')[0]
sp500tickers = list_slickcharts_sp500().index.tolist()

/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_17471/1385490009.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  return pd.read_html(response.text, match='Symbol', index_col='Symbol')[0]


In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}"}
base_url = "https://api.quiverquant.com/beta/historical/congresstrading/"
r = requests.get(base_url + "NVDA", headers=headers)
data = r.json()
df = pd.DataFrame(data)


=== NVDA ===
       Representative BioGuideID  ReportDate TransactionDate Ticker  \
0       Valerie Hoyle    H001094  2025-10-10      2025-09-23   NVDA   
1           Ro Khanna    K000389  2025-10-03      2025-09-05   NVDA   
2  Sheldon Whitehouse    W000802  2025-10-03      2025-09-04   NVDA   

      Transaction               Range            House   Amount Party  \
0            Sale  $50,001 - $100,000  Representatives  50001.0     D   
1            Sale    $1,001 - $15,000           Senate   1001.0     D   
2  Sale (Partial)   $15,001 - $50,000           Senate  15001.0     D   

  last_modified TickerType Description  ExcessReturn  PriceChange  SPYChange  
0    2025-10-10         ST        None      4.300788     7.319397   3.018609  
1    2025-10-06       None        None      8.761468    14.650940   5.889472  
2    2025-10-03      Stock        None      5.951744    11.551905   5.600161  

=== MSFT ===
         Representative BioGuideID  ReportDate TransactionDate Ticker  \
0    

In [ ]:
from time import sleep

base_url = "https://api.quiverquant.com/beta/historical/congresstrading/"
all_data = []

for i, ticker in enumerate(sp500tickers):
    url_t = base_url + ticker
    r = requests.get(url_t, headers=headers)

    if r.status_code == 200:
        try:
            data = r.json()
            if data:
                for row in data:
                    all_data.append({
                        "Ticker": ticker,
                        "Representative": row.get("Representative"),
                        "Party": row.get("Party"),
                        "House": row.get("House"),
                        "BioGuideID": row.get("BioGuideID")
                    })
        except Exception as e:
            print("JSON hatası:", ticker, e)
    else:
        print("Hata:", ticker, r.status_code)
    sleep(0.5)

print(f"\nToplam {len(all_data)} işlem kaydı çekildi.")
# === 4) DataFrame oluştur ===
df = pd.DataFrame(all_data)

# === 5) Unique temsilciler ===
distinct_df = df.drop_duplicates(subset=["Representative", "Party", "House", "BioGuideID"]).reset_index(drop=True)

print("\n=== Unique Congress Traders ===")
print(distinct_df.head(20))

Hata: CRWD 429
Hata: PLD 500
Hata: COP 500
Hata: ABNB 500
Hata: RCL 500
Hata: PWR 500
Hata: CMG 500
Hata: LHX 500
Hata: AMP 500
